# Profitability

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
from typing import get_args

import pandas as pd

from thesis_project.profitability import (
    fixed_horizon,
    fixed_profit,
    settings,
)

# Main

## Fixed horizon

In [ ]:
horizons = [
    10,
    30,
    1 * 60,
    5 * 60,
    10 * 60,
    15 * 60,
    30 * 60,
    45 * 60,
    1 * 60**2,
    2 * 60**2,
    3 * 60**2,
    24 * 60**2,
    3 * 24 * 60**2,
    5 * 24 * 60**2,
]

summaries = []
for horizon in horizons:
    summary = fixed_horizon.load_experiment(
        TICKER,
        "quoted",
        "round",
        "level",
        horizon,
    )
    summaries.append(summary)

summary = pd.concat(summaries)

In [ ]:
assert isinstance(summary, pd.DataFrame)
summary = summary.drop(
    columns=[
        "n_days",
        "trading_days",
        "total_pnl",
        "mean_pnl",
        "median_pnl",
        "std_pnl",
        "max_pnl",
        "min_pnl",
        "profitable_min_pnl",
        "episode_min_pnl",
        "min_episode_length",
    ]
)
summary = summary.rename(columns={col: col.replace("_", " ") for col in summary.columns})
summary = summary.rename(
    columns={c: c.replace("pct", "(%)") if "pct" in c else c for c in summary.columns}
)
summary = summary.rename(index=lambda x: pd.Timedelta(x, unit="s"), level="horizon")
display(summary.round(2))

## Fixed profit

In [ ]:
profits = [
    100.0,
    500.0,
    1_000.0,
    2_500.0,
    5_000.0,
    10_000.0,
    25_000.0,
    50_000.0,
]

summaries = {}
for price_mode in get_args(settings.PriceMode):
    for contract_mode in get_args(settings.FutContractMode):
        for volume_mode in ("ignore", "level"):
            data = []
            for profit in profits:
                summary = fixed_profit.load_experiment(
                    "fbtp",
                    price_mode,
                    contract_mode,
                    volume_mode,
                    profit,
                )
                data.append(summary)
            summaries[(price_mode, contract_mode, volume_mode)] = pd.concat(data)

In [ ]:
for k, v in summaries.items():
    print(k)
    v = v.drop(columns={"candidate_entries"})
    v = v.rename(columns={c: c.replace("_", " ") for c in v.columns})
    display(v)